# M1 — Three Modes of Complexity Proof (UFC/BJJ edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paiml/big-o-python-to-rust/blob/main/notebooks/m1-modes.ipynb)

Every Big-O claim ships with three independent receipts: empirical (criterion / `timeit`), structural (recurrence + proptest), formal (Lean 4 theorem). This notebook walks all three on a UFC roster, then introduces `depyler` as the transpiler shortcut that produces a starting-point Rust translation you then refine through the three receipts. Covers course lessons 1.1.1 (what complexity means), 1.2.1 (falsifiability), 1.3.1 (depyler shortcut).

## Lesson 1.1.1 — Empirical receipt: doubling-ratio check

Given mean times across doubling roster sizes, the max consecutive ratio reveals
the asymptotic class. For O(n) we expect a ratio close to 2. UFC rosters scale
from a handful (a small promotion) to hundreds (UFC + Bellator + ONE Championship
combined).

In [1]:
def empirical_doubling_ratio(times: list[float]) -> float:
    if len(times) < 2:
        return 0.0
    return max(times[i + 1] / times[i] for i in range(len(times) - 1))


# Synthetic O(n) doubling: 1ms at roster=1024, 2ms at 2048, 4ms at 4096, ...
times_linear = [1.0, 2.0, 4.0, 8.0, 16.0]
ratio = empirical_doubling_ratio(times_linear)
assert ratio == 2.0, f"expected ratio 2.0, got {ratio}"

# O(1) — ratios near 1 (Elo lookup is independent of roster size)
times_const = [1.0, 1.02, 0.99, 1.01]
assert empirical_doubling_ratio(times_const) < 1.1

print(f"empirical    : max doubling ratio = {ratio:.3f} (expected ~ 2.0 for O(n))")

empirical    : max doubling ratio = 2.000 (expected ~ 2.0 for O(n))


## Lesson 1.2.1 — Reading a complexity claim (falsifiability)

A complexity claim is falsifiable: it predicts a ratio bound, and you compute the
ratio. If the prediction fails, the claim falls. Three signals every reader checks:
input sizes form a doubling sequence; the ratio table is shown; the fit-versus-
alternative R² is named.

In [2]:
def is_well_formed_claim(input_sizes: list[int], ratios: list[float]) -> bool:
    """A complexity claim is well-formed iff sizes double and ratios are reported."""
    if len(input_sizes) < 2 or len(ratios) != len(input_sizes) - 1:
        return False
    return all(
        input_sizes[i + 1] == 2 * input_sizes[i] for i in range(len(input_sizes) - 1)
    )


# A well-formed O(n) claim: sizes double, ratios near 2
assert is_well_formed_claim([1024, 2048, 4096, 8192], [2.01, 1.99, 2.02])
# Malformed — sizes don't double
assert not is_well_formed_claim([1024, 1500, 2000], [1.5, 1.3])
print("falsifiable claim shape: doubling sizes + ratio table = OK")

falsifiable claim shape: doubling sizes + ratio table = OK


## Lesson 1.3.1 — `depyler` as the transpiler shortcut

`depyler transpile fighters.py` produces a starting-point Rust translation. The
three modes of proof then verify the transpile preserves the complexity class:
empirical bench shows the same doubling behavior, proptest locks the bound
structurally, the bound Lean theorem can be referenced.

You don't trust depyler; you measure it.

In [3]:
def transpile_class_preserved(py_class: str, rs_class: str) -> bool:
    """depyler's transpile-preservation contract: same complexity class on both sides."""
    return py_class == rs_class


def speedup(py_time: float, rs_time: float) -> float:
    return py_time / rs_time


# depyler keeps the class, but expects a constant-factor speedup on Rust side
assert transpile_class_preserved("O(n)", "O(n)")
assert speedup(40.0, 10.0) == 4.0  # 4x typical for naive list-comp -> iterator
print(
    "depyler contract: class preserved, expected speedup ~4x on iterator translations"
)

depyler contract: class preserved, expected speedup ~4x on iterator translations


## Structural — master-theorem case classification

The recurrence `T(n) = a * T(n/b) + f(n)` falls into one of three master-theorem
cases. A single-elimination tournament-bracket build is `T(n) = 2 T(n/2) + O(n)` =
Case 2 (O(n log n)). Knowing the case is the structural receipt.

In [4]:
from math import log


def classify_recurrence(a: float, b: float, f_exponent: float) -> str:
    """Return 'Case1', 'Case2', or 'Case3' for T(n) = a*T(n/b) + n^f_exponent."""
    critical = log(a, b)
    if f_exponent < critical:
        return "Case1"
    if f_exponent == critical:
        return "Case2"
    return "Case3"


# tournament bracket build: 2 T(n/2) + O(n) -> Case 2 = O(n log n)
assert classify_recurrence(2, 2, 1.0) == "Case2"
# binary search through ranked roster: 1 T(n/2) + 1 -> Case 2 = O(log n)
assert classify_recurrence(1, 2, 0.0) == "Case2"
# Strassen-ish: 8 T(n/2) + n^2 -> Case 1
assert classify_recurrence(8, 2, 2.0) == "Case1"

print("structural   : tournament-bracket recurrence in case 2 -> O(n log n)")

structural   : tournament-bracket recurrence in case 2 -> O(n log n)


## Formal — Lean theorem status enum

Empirical contracts (the 6 complexity classes) use `status: not-applicable` for
their Lean field because criterion + statistical CIs are the verification
mechanism. Master theorem and Fibonacci closed form are genuinely Lean-provable
(`status: proved`).

In [5]:
from enum import Enum


class ProofStatus(Enum):
    EMPIRICAL = "empirical"
    STRUCTURAL = "structural"
    FORMAL = "formal"


def applicable_mode(contract: str) -> ProofStatus:
    """Pick the natural proof mode for a contract."""
    if "master-theorem" in contract or "fibonacci-closed-form" in contract:
        return ProofStatus.FORMAL
    if "recurrence" in contract or "amortized" in contract:
        return ProofStatus.STRUCTURAL
    return ProofStatus.EMPIRICAL


assert applicable_mode("complexity-linear-v1") == ProofStatus.EMPIRICAL
assert applicable_mode("master-theorem-case-2") == ProofStatus.FORMAL
assert applicable_mode("banker-amortized-push") == ProofStatus.STRUCTURAL

print("formal       : status enum drives contract -> mode dispatch")

formal       : status enum drives contract -> mode dispatch


---
**Rust port:** [`m1-modes/src/lib.rs`](../m1-modes/src/lib.rs) implements the same functions. The Rust version exits with `contract: m1-modes-tour holds — OK` when every mode reports correctly. Course lessons 1.1.1, 1.2.1, 1.3.1.